# Employee Dataset Cleaning

This notebook demonstrates the cleaning process for the file `Messy_Employee_dataset.csv`.

Main steps:
- Load the raw dataset
- Split combined columns
- Handle missing values
- Clean numeric columns
- Convert date columns
- Remove duplicates
- Export the cleaned dataset


In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("Messy_Employee_dataset.csv")

print("Dataset loaded successfully.")
df.head()


Dataset loaded successfully.


,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


## Initial inspection

We first inspect the structure of the dataset, data types, and missing values.


In [2]:
print("Basic info:")
df.info()

print("\nMissing values per column:")
print(df.isna().sum())


Basic info:
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 88.8 KB

Missing values per column:
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0

## Split Department_Region into Department and Region

The column `Department_Region` contains two pieces of information separated by a dash (e.g., `Sales-East`).  
We split it into two new columns: `Department` and `Region`.


In [3]:
df[["Department", "Region"]] = df["Department_Region"].str.split("-", expand=True)

print("Department_Region successfully split into Department and Region.")
df[["Department", "Region"]].head()


Department_Region successfully split into Department and Region.


,Department,Region
0,DevOps,California
1,Finance,Texas
2,Admin,Nevada
3,Admin,Nevada
4,Cloud Tech,Florida


## Clean Age and Salary columns

We fill missing values in numeric columns `Age` and `Salary` using the median, which is robust to outliers.


In [4]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

print("Missing values after filling Age and Salary:")
print(df[["Age", "Salary"]].isna().sum())


Missing values after filling Age and Salary:
Age       0
Salary    0
dtype: int64


## Convert Join_Date to datetime

We convert the `Join_Date` column to datetime format.  
Invalid or unparseable dates will be converted to `NaT` using `errors="coerce"`.


In [5]:
df["Join_Date"] = pd.to_datetime(df["Join_Date"], errors="coerce")

print("Join_Date conversion complete. Sample values:")
df["Join_Date"].head()


Join_Date conversion complete. Sample values:


0   2021-04-02
1   2020-07-10
2   2023-12-07
3   2021-11-27
4   2022-01-05
Name: Join_Date, dtype: datetime64[us]

## Fix negative phone numbers

Some phone numbers may be negative.  
We convert them to their absolute values.


In [6]:
df["Phone"] = df["Phone"].abs()

print("Phone numbers cleaned. Sample values:")
df["Phone"].head()


Phone numbers cleaned. Sample values:


0    1651623197
1    1898471390
2    5596363211
3    3476490784
4    1586734256
Name: Phone, dtype: int64

## Remove duplicate employees based on Email

We assume `Email` uniquely identifies an employee.  
We remove duplicate rows based on the `Email` column, keeping the first occurrence.


In [7]:
before = len(df)
df = df.drop_duplicates(subset="Email", keep="first")
after = len(df)

print(f"Removed {before - after} duplicate rows.")
print(f"Final number of rows: {after}")


Removed 956 duplicate rows.
Final number of rows: 64


## Reset index and drop old combined column

After cleaning, we reset the index and drop the original `Department_Region` column, since it has been split.


In [8]:
df = df.reset_index(drop=True)
df = df.drop(columns=["Department_Region"])

print("Index reset and Department_Region column dropped.")
df.head()


Index reset and Department_Region column dropped.


,Employee_ID,First_Name,Last_Name,Age,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region
0,EMP1000,Bob,Davis,25.0,Active,2021-04-02,59767.65,bob.davis@example.com,1651623197,Average,True,DevOps,California
1,EMP1001,Bob,Brown,30.0,Active,2020-07-10,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,30.0,Pending,2023-12-07,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25.0,Inactive,2021-11-27,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25.0,Active,2022-01-05,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


## Save cleaned dataset

We save the cleaned dataset as `Clean_Employee_Dataset.csv`.


In [9]:
df.to_csv("Clean_Employee_Dataset.csv", index=False)

print("Cleaning complete. File saved as Clean_Employee_Dataset.csv")


Cleaning complete. File saved as Clean_Employee_Dataset.csv
